In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso

from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from sklearn.metrics import (
    mean_squared_error,
    r2_score
)

import joblib

cargamos los datos limpios

In [27]:
X = pd.read_csv(
    "00.data/cleaned/X_scaled.csv"
)

test = pd.read_csv(
    "00.data/cleaned/test_scaled.csv"
)

y = pd.read_csv(
    "00.data/cleaned/y.csv"
)

La variable objetivo se convierte en una estructura unidimensional para facilitar el entrenamiento de los modelos.

In [28]:
y = y.squeeze()

el dataset de train se divide en entrenamiento y validacion

In [29]:
X_train, X_valid, y_train, y_valid = train_test_split(
    
    X,
    y,
    
    test_size=0.2,
    
    random_state=42
)

Se utiliza una regresión lineal como modelo base para obtener una primera referencia 

In [30]:
lr = LinearRegression()

lr.fit(X_train, y_train)

pred_lr = lr.predict(X_valid)

In [31]:
rmse_lr = np.sqrt(
    mean_squared_error(
        y_valid,
        pred_lr
    )
)

r2_lr = r2_score(
    y_valid,
    pred_lr
)

print("Linear Regression RMSE:", rmse_lr)

print("Linear Regression R2:", r2_lr)

Linear Regression RMSE: 0.13714374409363772
Linear Regression R2: 0.8884279072899963


utilizamos rmse y el r2 como metricas principales porque una mide el error medio de prediccion y la otra mide la capacidad del modelo para explicar la variabilidad de la variable objetivo 

usamos tambien ridge regression para reducir posibles problemas de sobreajuste y mejorar estabilidad    

In [32]:
ridge = Ridge(alpha=10)

ridge.fit(X_train, y_train)

pred_ridge = ridge.predict(X_valid)
rmse_ridge = np.sqrt(
    mean_squared_error(
        y_valid,
        pred_ridge
    )
)

r2_ridge = r2_score(
    y_valid,
    pred_ridge
)

print("Ridge RMSE:", rmse_ridge)

print("Ridge R2:", r2_ridge)

Ridge RMSE: 0.13241531436030624
Ridge R2: 0.8959888238593352


Lasso Regression también utiliza regularización y puede reducir la importancia de variables menos relevantes

In [33]:
lasso = Lasso(alpha=0.001)

lasso.fit(X_train, y_train)

pred_lasso = lasso.predict(X_valid)
rmse_lasso = np.sqrt(
    mean_squared_error(
        y_valid,
        pred_lasso
    )
)

r2_lasso = r2_score(
    y_valid,
    pred_lasso
)

print("Lasso RMSE:", rmse_lasso)

print("Lasso R2:", r2_lasso)

Lasso RMSE: 0.1285672881448871
Lasso R2: 0.9019461743961381


Random Forest combina múltiples árboles de decisión para mejorar la capacidad predictiva y reducir el riesgo de sobreajuste.

In [34]:
rf = RandomForestRegressor(
    
    n_estimators=200,
    
    random_state=42
)

rf.fit(X_train, y_train)

pred_rf = rf.predict(X_valid)
rmse_rf = np.sqrt(
    mean_squared_error(
        y_valid,
        pred_rf
    )
)

r2_rf = r2_score(
    y_valid,
    pred_rf
)

print("Random Forest RMSE:", rmse_rf)

print("Random Forest R2:", r2_rf)

Random Forest RMSE: 0.14661111083237094
Random Forest R2: 0.8724920263218983


XGBoost es un modelo basado en boosting que suele ofrecer muy buenos resultados en datasets tabulares como este.

In [35]:
xgb = XGBRegressor(
    
    n_estimators=1000,
    
    learning_rate=0.05,
    
    max_depth=3,
    
    subsample=0.8,
    
    colsample_bytree=0.8,
    
    random_state=42
)
xgb.fit(X_train, y_train)
pred_xgb = xgb.predict(X_valid)
rmse_xgb = np.sqrt(
    mean_squared_error(
        y_valid,
        pred_xgb
    )
)

r2_xgb = r2_score(
    y_valid,
    pred_xgb
)

print("XGBoost RMSE:", rmse_xgb)

print("XGBoost R2:", r2_xgb)

XGBoost RMSE: 0.12097304095774375
XGBoost R2: 0.9131877982082819


Se comparan los distintos modelos utilizando RMSE y R² como métricas principales para seleccionar el modelo con mejor rendimiento.

In [36]:
results = pd.DataFrame({
    
    "Model": [
        "Linear Regression",
        "Ridge",
        "Lasso",
        "Random Forest",
        "XGBoost"
    ],
    
    "RMSE": [
        rmse_lr,
        rmse_ridge,
        rmse_lasso,
        rmse_rf,
        rmse_xgb
    ],
    
    "R2": [
        r2_lr,
        r2_ridge,
        r2_lasso,
        r2_rf,
        r2_xgb
    ]
})

results.sort_values("RMSE")

,Model,RMSE,R2
4,XGBoost,0.120973,0.913188
2,Lasso,0.128567,0.901946
1,Ridge,0.132415,0.895989
0,Linear Regression,0.137144,0.888428
3,Random Forest,0.146611,0.872492


el mejor modelo es el XGBOOST y es seleccionado como modelo final para realizar predicciones en el dataset de test.

entreno el 100% de los datos 

In [37]:
xgb.fit(X, y)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [38]:
preds = xgb.predict(test)

Como durante el preprocessing se aplicó una transformación logarítmica sobre sale price , es necesario deshacer la transformación para obtener los precios reales.

In [39]:
preds = np.expm1(preds)

Se crea un archivo csv con los identificadores de las viviendas y las predicciones finales

In [40]:
test_original = pd.read_csv(
    "00.data/raw/test.csv"
)

In [41]:
submission = pd.DataFrame({
    
    "Id": test_original["Id"],
    
    "SalePrice": preds
})
submission.to_csv(
    "submission.csv",
    index=False
)

Finalmente, el mejor modelo entrenado se guarda para poder reutilizarlo posteriormente sin necesidad de volver a entrenarlo.

In [42]:
joblib.dump(
    xgb,
    "best_model.pkl"
)

['best_model.pkl']

Tras comparar distintos modelos, XGBoost obtuvo el mejor rendimiento, con el menor rmse y el mayor r2.

Esto indica que fue el modelo con mejor capacidad predictiva sobre el conjunto de validación.

Finalmente, entrenamos a XGBoost con todos los datos disponibles y se generó el archivo final de predicciones.

In [44]:
preds[:50]

array([124094.234, 162398.42 , 187272.34 , 194704.61 , 179114.83 ,
       172992.19 , 180574.06 , 168594.3  , 179990.53 , 125305.35 ,
       190406.42 ,  95585.586, 101096.33 , 149618.2  , 117675.37 ,
       376091.78 , 261777.27 , 271835.66 , 250637.16 , 503718.12 ,
       345653.1  , 205636.77 , 172845.42 , 166435.39 , 185334.83 ,
       191907.86 , 335868.12 , 230511.11 , 201069.05 , 216073.11 ,
       190012.61 ,  86520.63 , 188288.28 , 288421.22 , 268299.7  ,
       231969.25 , 182611.36 , 163724.28 , 159460.83 , 151013.6  ,
       175053.75 , 140163.72 , 298904.72 , 237674.62 , 226331.53 ,
       190084.56 , 250507.88 , 201718.42 , 155616.2  , 146392.22 ],
      dtype=float32)